# **ASSIGNMENT 1**

## Question 1

In [24]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json

BASE = "https://sandbox.oxylabs.io"

URL = "https://sandbox.oxylabs.io/products"
r = requests.get(URL)
print(r)

<Response [200]>


In [25]:
soup = BeautifulSoup(r.content, 'html.parser')

In [26]:
pages = soup.find_all('li', class_='next')

In [27]:
id_=[]
name=[]
status=[]
description=[]
detailUrl=[]
listingPageNo=[]
price=[]


In [28]:
page_no = 1
while True:
    r = requests.get(URL, params={"page": page_no})
    soup = BeautifulSoup(r.content, 'html.parser')
    products = soup.find_all('div', class_='product-card')
    if not products:
        break
    print("Page No:", page_no, "\tProducts:", len(products))

    next_data = json.loads(soup.find("script", id="__NEXT_DATA__").string)  # idk man i jst lowk guessed and re-guessed till it worked dont ask me questions abt this one. ig there's a json data stream thingamjiggi dont have time ton do a dep dive on json
    stock_lookup = {item["id"]: item["inStock"] for item in next_data["props"]["pageProps"]["products"]}  
    #print(stock_lookup)
    
    for i in range(len(products)):
        product = products[i]
        a = product.find("a", class_="card-header")
        href = a["href"] if a else ""
        id_.append(href.split("/")[-1])
        detailUrl.append(BASE + href)

        h4 = product.find("h4", class_="title")
        name.append(h4.get_text(strip=True) if h4 else "")

        status.append("In Stock" if stock_lookup.get(int(href.split("/")[-1])) else "Out of Stock")

        desc = product.find("p", class_="description")
        description.append(desc.get_text(strip=True) if desc else "")

        price_tag = product.find("div", class_="price-wrapper")
        price.append(price_tag.get_text(strip=True) if price_tag else "")

        listingPageNo.append(page_no)

    if soup.find("li", class_="next disabled"):
        break
    page_no += 1

Page No: 1 	Products: 32
Page No: 2 	Products: 32
Page No: 3 	Products: 32
Page No: 4 	Products: 32
Page No: 5 	Products: 32
Page No: 6 	Products: 32
Page No: 7 	Products: 32
Page No: 8 	Products: 32
Page No: 9 	Products: 32
Page No: 10 	Products: 32
Page No: 11 	Products: 32
Page No: 12 	Products: 32
Page No: 13 	Products: 32
Page No: 14 	Products: 32
Page No: 15 	Products: 32
Page No: 16 	Products: 32
Page No: 17 	Products: 32
Page No: 18 	Products: 32
Page No: 19 	Products: 32
Page No: 20 	Products: 32
Page No: 21 	Products: 32
Page No: 22 	Products: 32
Page No: 23 	Products: 32
Page No: 24 	Products: 32
Page No: 25 	Products: 32
Page No: 26 	Products: 32
Page No: 27 	Products: 32
Page No: 28 	Products: 32
Page No: 29 	Products: 32
Page No: 30 	Products: 32
Page No: 31 	Products: 32
Page No: 32 	Products: 32
Page No: 33 	Products: 32
Page No: 34 	Products: 32
Page No: 35 	Products: 32
Page No: 36 	Products: 32
Page No: 37 	Products: 32
Page No: 38 	Products: 32
Page No: 39 	Products

In [29]:
import pandas as pd

df = pd.DataFrame({
    "id": id_,
    "name": name,
    "status": status,
    "description": description,
    "detailUrl": detailUrl,
    "price": price,
    "listingPageNo": listingPageNo
})

df.to_csv("23L_0759_VersionA_static_products_products.csv", index=False)
print(df.shape)

(3000, 7)


In [30]:
import csv

with open("23L_0759_VersionA_static_products_products.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "name", "status", "description", "detailUrl", "price", "listingPageNo"])
    writer.writerows(zip(id_, name, status, description, detailUrl, price, listingPageNo))

### STATS:

In [48]:
print(f"Total pages processed: {page_no}")
print(f"Total records extracted: {len(id_)}")

complete_records = 0

for item in zip(id_, name, price, description, detailUrl, status):
    if not any(x is None or x == "" for x in item):
        complete_records += 1

incomplete_records = len(id_) - complete_records

print(f"Complete records (no missing fields): {complete_records}") #this seems wrong idk why dont have time
print(f"Incomplete records (at least one missing field): {incomplete_records}")

Total pages processed: 94
Total records extracted: 3000
Complete records (no missing fields): 3000
Incomplete records (at least one missing field): 0


### Writeup:
#### How you identified the relevant elements/records on each page (your general approach, not a line-by-line selector dump).
PRessed ctrl+shift+i to inspect element, hovered over the product cards, identified the name of the div. Hovered over each element, identified the type of element (p, div, or h4). Stock status was more annoying... it's nowhere in the rendered HTML. Looks like a placeholder. Looked it up, said to check the raw page source. Looked up the keyboard shortcut to check that. Pressed Ctrl+U, I checked the raw page source and found a ```<script id="__NEXT_DATA__">``` tag containing a JSON blob with all product data including an inStock boolean per product. I parsed that and just made a dict for it.
#### How you navigated across pages (pagination, scrolling, clicking, etc.).
Simple, every time you loop, you simply go to the next page by requesting it's url which is https://sandbox.oxylabs.io/products/?page=1, https://sandbox.oxylabs.io/products?page=2 and so on.
#### How your program decided that scraping was complete (i.e., how it knew there were no more pages/records).
Simple as well. The last page has ```li.next.disabled``` element. The loop checks for that after processing each page and breaks if found. There's also a fallback break if no product cards come back at all.
#### Any challenges you ran into and how you resolved them.
The stock alert was the hardest part, i spent a long while debugging that. Also I was writing the wrong values to the csv file so I was very confused why my program was wrong for a while.
#### How you verified that your scraper was actually collecting correct, complete data (spot checks, counts, etc.).
Ran a loop to count the pages and products on the pages


## QUESTION TWO

In [1]:
# Install Google Chrome
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb
!apt-get -f install -y
!yay -S google-chome


--2026-09-12 22:25:23--  https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving dl.google.com (dl.google.com)... 2a00:1450:4028:80a::200e, 142.250.75.206
Connecting to dl.google.com (dl.google.com)|2a00:1450:4028:80a::200e|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 141931876 (135M) [application/x-debian-package]
Saving to: ‘google-chrome-stable_current_amd64.deb.3’

google-chrome-stabl 100%[===================>] 135.36M  5.39MB/s    in 25s     

2026-09-12 22:25:49 (5.32 MB/s) - ‘google-chrome-stable_current_amd64.deb.3’ saved [141931876/141931876]

/usr/bin/bash: line 1: dpkg: command not found
/usr/bin/bash: line 1: apt-get: command not found
 -> No AUR package found for google-chome
 -> no package found for targets


In [2]:
# Install Selenium and ChromeDriver Manager
!pip install selenium webdriver-manager #This took me ONE HOUR to do so annoying

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try 'pacman -S
    python-xyz', where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Arch-packaged Python package,
    create a virtual environment using 'python -m venv path/to/venv'.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip.
    
    If you wish to install a non-Arch packaged Python application,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. Make sure you have python-pipx
    installed via pacman.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.


In [15]:
#######Selenium Functions & Methods######
#Navigation: get, backword,forward
#Element location: find_element(By.ID,By.NAME,By.XPATH etc)
#Element interation: click, send_keys,clear
#Element information: text, get_attribute, is_displayed
from selenium import webdriver
#Imports Selenium's WebDriver, which allows Python to control Chrome.
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
#By is used later to find elements on a webpage.
#By.ID
#By.CLASS_NAME
#By.XPATH
#By.CSS_SELECTOR
from selenium.webdriver.common.by import By
#--------------------------#
# Set up Chrome options
options = webdriver.ChromeOptions()
options.add_argument('--headless') # Run in headless mode (Chrome runs without displaying the browser window)
#Python → Chrome runs in background → Website
#Helps Chrome run in some restricted environments
options.add_argument('--no-sandbox')
#Helps prevent certain memory-related problems when Chrome runs in limited environments.
options.add_argument('--disable-dev-shm-usage')
options.binary_location = '/usr/bin/chromium' # Set the Chrome binary location
#--------------------------#
# Set up ChromeDriver service
#ChromeDriver acts as a bridge between Python/Selenium and Chrome:
#Python -> Selenium -> ChromeDriver -> Google Chrome -> Website
service = Service('/usr/bin/chromedriver')
# Initialize the WebDriver with Chrome
#This finally starts Chrome and connects it to Selenium.


# Initialize WebDriver
driver = webdriver.Chrome(service=service, options=options)

In [16]:
url = "https://www.scrapingcourse.com/infinite-scrolling"
driver.get(url)

# Wait until products are loaded
wait = WebDriverWait(driver, 10)

wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'a[href*="/ecommerce/product/"]')
    )
)

print("Page loaded")

Page loaded


In [17]:
url = "https://www.scrapingcourse.com/infinite-scrolling"
driver.get(url)

wait = WebDriverWait(driver, 10)

wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'a[href*="/ecommerce/product/"]')
    )
)

print("Page loaded")

Page loaded


In [18]:
product_details = []

products = driver.find_elements(
    By.CSS_SELECTOR,
    'a[href*="/ecommerce/product/"]'
)

print("Products before scrolling:", len(products))

Products before scrolling: 12


In [19]:
import re
for product in products:

    name = product.text.strip()

    price = ""
    if "$" in name:
        price = re.search(r'\$[\d,.]+', name).group()

    name = re.sub(r'\$[\d,.]+', '', name).strip()

    detail_url = product.get_attribute("href")

    # Get image URL
    image_url = ""
    try:
        image = product.find_element(By.TAG_NAME, "img")
        image_url = image.get_attribute("src")
    except:
        image_url = ""

    product_details.append({
        "Product name": name,
        "Price": price,
        "Image URL": image_url,
        "Scroll batch": 0,
        "Detail URL": detail_url
    })

In [20]:
scroll_batch = 0

while True:
    old_count = len(
        driver.find_elements(
            By.CSS_SELECTOR,
            'a[href*="/ecommerce/product/"]'
        )
    )
    driver.execute_script(
        "window.scrollTo(0, document.documentElement.scrollHeight);"
    )

    scroll_batch += 1

    try:
        wait.until(
            lambda d: len(
                d.find_elements(
                    By.CSS_SELECTOR,
                    'a[href*="/ecommerce/product/"]'
                )
            ) > old_count
        )
    except:
        print("No new products loaded.")
        break

    products = driver.find_elements(
        By.CSS_SELECTOR,
        'a[href*="/ecommerce/product/"]'
    )

    print(
        "Scroll batch:",
        scroll_batch,
        "Total products:",
        len(products)
    )

    for product in products:

        detail_url = product.get_attribute("href")

        already_exists = False

        for item in product_details:
            if item["Detail URL"] == detail_url:
                already_exists = True
                break

        if already_exists == False: #no duplictyes

            name = product.text.strip()

            price = ""
            if "$" in name:
                price = re.search(r'\$[\d,.]+', name).group()

            name = re.sub(r'\$[\d,.]+', '', name).strip()

            image_url = ""

            try:
                image = product.find_element(By.TAG_NAME, "img")
                image_url = image.get_attribute("src")
            except:
                image_url = ""

            product_details.append({
                "Product name": name,
                "Price": price,
                "Image URL": image_url,
                "Scroll batch": scroll_batch,
                "Detail URL": detail_url
            })

print("Total unique products:", len(product_details))

Scroll batch: 1 Total products: 24
Scroll batch: 2 Total products: 36
Scroll batch: 3 Total products: 48
Scroll batch: 4 Total products: 60
Scroll batch: 5 Total products: 72
Scroll batch: 6 Total products: 84
Scroll batch: 7 Total products: 96
Scroll batch: 8 Total products: 108
Scroll batch: 9 Total products: 120
Scroll batch: 10 Total products: 132
Scroll batch: 11 Total products: 144
Scroll batch: 12 Total products: 156
Scroll batch: 13 Total products: 168
Scroll batch: 14 Total products: 180
Scroll batch: 15 Total products: 187
No new products loaded.
Total unique products: 147


In [49]:
i = 0
for item in product_details: #this takes FORECVERRRRRRRR to run no way this is good
    print(i)
    r = requests.get(item["Detail URL"])
    soup = BeautifulSoup(r.content, 'html.parser')

    sku_el = soup.find("span", class_="sku")
    item["SKU"] = sku_el.get_text(strip=True) if sku_el else ""

    desc_el = soup.find("div", class_="woocommerce-product-details__short-description")
    item["Short Description"] = desc_el.get_text(strip=True) if desc_el else ""
    i = i + 1

driver.quit()
print("done:", len(product_details))

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
done: 147


In [22]:
df = pd.DataFrame(product_details)
df = df[["Product name", "Price", "Image URL", "Scroll batch", "SKU", "Short Description", "Detail URL"]]
df.to_csv("23L_0759_VersionA_dynamic_products.csv", index=False)
print(df.shape)

(147, 7)


In [52]:
## Validation

In [50]:
print(f"Scroll batches completed: {scroll_batch}")
print(f"Total unique products collected: {len(product_details)}")

batch_0 = sum(1 for x in product_details if x["Scroll batch"] == 0)
print(f"Products present before any scroll (batch 0): {batch_0}")
print(f"Products loaded via scrolling (batch 1+): {len(product_details) - batch_0}")

complete_q2 = sum(
    1 for x in product_details
    if x.get("Product name") and x.get("Price") and x.get("SKU") and x.get("Short Description")
)
print(f"Records with all required fields populated: {complete_q2}")
print(f"Records with at least one missing field: {len(product_details) - complete_q2}")

=== Q2 Validation Stats ===
Scroll batches completed: 16
Total unique products collected: 147
Products present before any scroll (batch 0): 12
Products loaded via scrolling (batch 1+): 135
Records with all required fields populated: 143
Records with at least one missing field: 4


In [62]:
## Writeup:
#### How you identified the relevant elements/records on each page (your general approach, not a line-by-line selector dump).
Again, pessed ```ctrl+shift+i``` to inspect element, hovered over the product cards all product tiles on the listing page are ```<a>``` tags whose `href` contains ```/ecommerce/product/```. The anchors .text renders both the product name and price as a combined string, so I used regex to extract the dollar amount and then stripped it from the name. The product image is nested inside the same anchor, so find_element(By.TAG_NAME, "img") grabs it. For detail pages, fetched separately with requests + BeautifulSoup since theyre static HTML and I didnt want to mess with the selenium scrolling by doing both with selenium.
#### How you navigated across pages (pagination, scrolling, clicking, etc.).
Used `driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight)")` to jump to the very bottom of the page each iteration, triggering the site infinite-scroll loader. Each execution of this counts as one scroll batch. which i counted

### How you handled dynamic content in Question 2, including any waits you used and why.
Instead of `time.sleep()` like in class, I used `WebDriverWait` with a lambda that compares the current product count against the count recorded before the scroll. As the count goes up,the loop moves as new content is in the DOM rather than waiting a fixed amount of time regardless. Fixed sleeps would either be too slow (wasting time) or too fast (missing content on a slow connection). And also being honest, it was what was required in the assignment so I couldnt use sleep.

#### How your program decided that scraping was complete (i.e., how it knew there were no more pages/records).
When `wait.until()` times out, meaning no new products appeared within the 10-second window after a scroll. This breaks the loop in an except block.

#### Any challenges you ran into and how you resolved them.
Honestly, the imports were the hardest part of this that took the longest because I hadnt done it before and collecting all the libraries and checking all the file paths was a chore. Also, figuring out how to wait and dynamic scraping in general from scratch was pretty hard too. Didnt leavt time for the bonus.
#### How you verified that your scraper was actually collecting correct, complete data (spot checks, counts, etc.).
Counted the number of batches and then counted the number of detail pages in total. Also, made sure the number of unique products matcht eh number of detail pages. (147 currently

SyntaxError: invalid syntax (4029093499.py, line 3)